# 007 · Mutability and Identity

> **Syntax › Data Types** · Python 3.14

In C# the first question about a type is "value type or reference type?". Python has no value types at all — **everything is an object, and every variable is a reference**. The question that matters instead is: *is the object mutable or immutable?*

**In this notebook you will learn:**

- that every value is an object with an identity (`id()`), a type, and a value
- which built-in types are mutable and which are immutable
- the difference between *rebinding* a name and *mutating* an object
- how arguments are passed ("pass by object reference") compared to C#'s value/`ref`/`out`
- why dictionary keys and set members must be hashable (immutable)
- `==` vs `is`, and shallow vs deep copies

## C# mental model

| C# | Python |
|---|---|
| Value types (`int`, `struct`, `DateTime`) — copied on assignment | No value types. Every variable holds a reference |
| Reference types (`class`, `List<T>`) — reference copied | Same for **all** objects |
| Immutable `string`, `record`, `readonly struct` | Immutable `int`, `float`, `str`, `tuple`, `frozenset`, ... |
| `object.ReferenceEquals(a, b)` | `a is b` |
| `a.Equals(b)` / `==` | `a == b` |
| `ref` / `out` parameters | none — return a value (or a tuple) instead |
| `GetHashCode()` | `hash(x)` / `__hash__` |

Immutable objects behave a lot like C# value types in practice: since nobody can change them, sharing a reference is indistinguishable from copying.

## Everything is an object

Ints, functions, classes, modules, even `None` — all are objects with an identity, a type, and a value. `id(x)` returns the identity (in CPython, the memory address), which never changes during the object's lifetime.

In [ ]:
n = 42
print(type(n), id(n))
print(n.bit_length())             # even an int literal has methods
print((3.5).is_integer())

def greet(): ...
print(type(greet), type(int), type(None))   # functions and classes are objects too
print(isinstance(len, object), isinstance(int, object))

## Names are references

Assignment never copies an object — it binds a **name** to an object. Two names can refer to the same object. Think of every Python variable as a C# reference-type variable.

In [ ]:
a = [1, 2, 3]
b = a                     # b refers to the SAME list
print(a is b, id(a) == id(b))

b.append(4)
print(a)                  # [1, 2, 3, 4] — visible through both names

## Mutable vs immutable types

| Immutable (can't change after creation) | Mutable (can change in place) |
|---|---|
| `int`, `float`, `complex`, `bool`, `Decimal`, `Fraction` | `list` |
| `str`, `bytes` | `bytearray` |
| `tuple` | `dict` |
| `frozenset` | `set` |
| `None`, `range` | most user-defined classes (by default) |

Immutable types have no methods that change them — every "modifying" operation returns a **new** object.

In [ ]:
s = "hello"
t = s.upper()               # new object; s is untouched
print(s, t)

nums = (1, 2, 3)
try:
    nums[0] = 99
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Rebinding vs mutating

This is *the* key distinction:

- **Rebinding** makes a name point to a different object (`x = ...`, and `x += ...` on immutables). Other names are unaffected.
- **Mutating** changes the object itself (`.append()`, `[i] = ...`, and `+=` on lists). Every name referring to it sees the change.

In [ ]:
# Immutable: += creates a new object and rebinds the name
x = 10
y = x
x += 1
print(x, y)                 # 11 10 — y still points at the old int

s = "ab"
before = id(s)
s += "c"
print(id(s) == before)      # False — a new string was created

In [ ]:
# Mutable: += mutates the list in place (it calls list.extend)
items = [1, 2]
alias = items
before = id(items)
items += [3]
print(id(items) == before)  # True — same object
print(alias)                # [1, 2, 3] — alias sees it

# ...whereas + creates a new list and rebinds
items = items + [4]
print(items, alias)         # [1, 2, 3, 4] [1, 2, 3]

⚠️ **Gotcha:** `x += y` means "mutate if you can, otherwise rebind". So the same line of code behaves differently for a `list` (mutates) and a `tuple` (rebinds).

## No value types — even "small" things are references

In C#, `struct Point` is copied on assignment. In Python, a user-defined class instance is always shared by reference, so you get "struct-like" behaviour only by making the type immutable (e.g. a `tuple`, `NamedTuple`, or a frozen dataclass — *see Classes & OOP › 008 · Dataclasses*).

```csharp
// C#
struct Point { public int X; public int Y; }
var p1 = new Point { X = 1, Y = 2 };
var p2 = p1;      // copy!
p2.X = 99;
Console.WriteLine(p1.X);   // 1
```

In [ ]:
class Point:
    def __init__(self, x: int, y: int):
        self.x, self.y = x, y

p1 = Point(1, 2)
p2 = p1                 # NOT a copy — same object
p2.x = 99
print(p1.x)             # 99 — would be 1 with a C# struct

## Passing arguments: "pass by object reference"

Python passes a **reference to the object** — not a copy of the value, and not a reference to the caller's variable. This is exactly how C# passes reference types *without* `ref`:

- If the function **mutates** the object, the caller sees it.
- If the function **rebinds** its parameter, the caller does not see it.

There is no `ref` or `out`. To "return several things", return a tuple *(see Collections › 002 · Tuples)*.

```csharp
// C#
void AddItem(List<int> list) => list.Add(99);              // caller sees it
void Replace(List<int> list) => list = new List<int>();    // caller does not
void ReplaceRef(ref List<int> list) => list = new();       // caller does — no Python equivalent
```

In [ ]:
def add_item(lst: list[int]) -> None:
    lst.append(99)          # mutation — visible to caller

def replace(lst: list[int]) -> None:
    lst = [0, 0, 0]         # rebinding the local name only

def increment(n: int) -> None:
    n += 1                  # ints are immutable: this rebinds locally

data = [1, 2]
add_item(data)
replace(data)
print(data)                 # [1, 2, 99]

count = 5
increment(count)
print(count)                # 5 — unchanged

The Pythonic replacement for `ref`/`out` is to **return** the new value(s):

```csharp
// C#
bool TryDivide(int a, int b, out int result) { ... }
```

In [ ]:
def divide(a: int, b: int) -> tuple[int, int]:
    return a // b, a % b       # return multiple values as a tuple

quotient, remainder = divide(17, 5)
print(quotient, remainder)

## `==` vs `is`

- `a == b` compares **values** (calls `__eq__`) — like `Equals()`.
- `a is b` compares **identity** — like `ReferenceEquals(a, b)`.

Unlike C#, where `==` on a class defaults to reference comparison, Python's built-in collections compare by value with `==`.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(a == b, a is b)    # True False — equal values, different objects
print(a == c, a is c)    # True True

⚠️ **Gotcha:** never use `is` to compare numbers or strings. CPython caches small ints (−5 to 256) and some strings, so `is` *sometimes* appears to work — and then fails for other values. Use `is` only for `None`, `True`/`False` singletons, and genuine identity checks.

In [ ]:
x = int("100")
y = int("100")
print(x is y)            # True  — small ints are cached (implementation detail!)

x = int("1000")
y = int("1000")
print(x == y, x is y)    # True False — two different int objects

## Hashability: why dict keys must be immutable

Dictionaries and sets locate items by **hash**. If a key could change after being inserted, its hash would change and the dict could never find it again. So Python only allows **hashable** objects as keys and set members — in practice, immutable ones.

C# allows mutable keys and simply trusts you not to mutate them; Python enforces it for built-in types.

In [ ]:
locations = {(0, 0): "origin", (1, 2): "shop"}   # tuples are hashable
print(locations[(1, 2)])

try:
    bad = {[0, 0]: "origin"}                      # lists are not
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(hash("abc") == hash("abc"), hash(42))

A tuple is hashable only if everything inside it is hashable. Use `frozenset` for an immutable, hashable set:

In [ ]:
try:
    hash((1, [2, 3]))
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

tags = {frozenset({"red", "large"}): "SKU-1"}
print(tags[frozenset({"large", "red"})])        # order doesn't matter

User-defined class instances are hashable by **identity** by default (like C#'s default `GetHashCode`). If you define `__eq__`, Python sets `__hash__` to `None` — making the class unhashable — unless you also define `__hash__`. *(Classes & OOP › 007 · Dunder Methods covers this.)*

In [ ]:
class Product:
    def __init__(self, sku: str):
        self.sku = sku
    def __eq__(self, other):
        return isinstance(other, Product) and self.sku == other.sku

try:
    {Product("A1")}
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Copying: shallow vs deep

Since assignment never copies, you copy explicitly:

- **Shallow copy** — a new outer container holding the *same* inner objects: `lst.copy()`, `list(lst)`, `dict(d)`, `copy.copy(x)`.
- **Deep copy** — recursively copies everything: `copy.deepcopy(x)`.

*(Collections › 001 · Lists goes into more detail.)*

In [ ]:
import copy

order = {"id": 1, "items": ["pen", "cup"]}

shallow = copy.copy(order)
deep = copy.deepcopy(order)

order["items"].append("lamp")     # mutate the shared inner list
order["id"] = 2                   # rebind a key in the original only

print(shallow)    # {'id': 1, 'items': ['pen', 'cup', 'lamp']} — inner list shared
print(deep)       # {'id': 1, 'items': ['pen', 'cup']} — fully independent

Copying immutable objects is pointless — `copy.copy` on a `str` or `tuple` of immutables just returns the same object:

In [ ]:
t = (1, 2, 3)
print(copy.copy(t) is t, copy.deepcopy(t) is t)   # True True

## Summary

| Concept | C# | Python |
|---|---|---|
| Value semantics | `struct` copied on assignment | none — use immutable types instead |
| Variable | value or reference | always a reference |
| Identity | `ReferenceEquals(a, b)` | `a is b` / `id(a) == id(b)` |
| Equality | `Equals` / `==` | `==` |
| Mutate argument | works for reference types | works for mutable objects |
| Rebind caller's variable | `ref` / `out` | not possible — return a value |
| Immutable text | `string` | `str` |
| Immutable collection | `ImmutableList<T>`, `IReadOnlyList<T>` | `tuple`, `frozenset` |
| Hash code | `GetHashCode()` | `hash(x)` |
| Dict key requirement | any (be careful) | must be hashable |
| Shallow / deep copy | `MemberwiseClone` / manual | `copy.copy` / `copy.deepcopy` |

## Exercises

1. Predict the output, then run it:
   ```python
   a = (1, [2, 3])
   b = a
   a[1].append(4)
   print(b)
   ```
   Is the tuple mutable? Explain.
2. Write a function `with_item(items, item)` that returns a **new** list with `item` appended, leaving the caller's list untouched.
3. Build a dict that maps a 2-D grid coordinate to a cell value, so that `grid[(2, 3)]` works. Why can't you use `[2, 3]` as the key?
4. Why does `count = 0; bump(count)` not change `count` even if `bump` does `n += 1`, while `bump_all(lst)` doing `lst.append(1)` does change the caller's list?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — prints (1, [2, 3, 4]). The tuple itself is immutable (it always holds the
#     same two references), but one of those references points to a mutable list.
a = (1, [2, 3])
b = a
a[1].append(4)
print(b)

# 2
def with_item(items: list, item) -> list:
    return [*items, item]      # or: items + [item]

original = [1, 2]
print(with_item(original, 3), original)   # [1, 2, 3] [1, 2]

# 3 — tuples are hashable; lists are mutable and therefore unhashable
grid = {(x, y): x * y for x in range(4) for y in range(4)}
print(grid[(2, 3)])   # 6

# 4 — ints are immutable: `n += 1` creates a new int and rebinds the
#     local name n. lst.append mutates the shared list object itself.
```

</details>